# [INFO] Glu-Stock: 03_EXECUTION_MONITOR
**Phase**: Autonomous Execution & Global Monitoring (v18.3)

This unified notebook manages trade execution (ATR sizing), monitors open positions (SL/TP), and sends real-time Telegram reports.

In [ ]:
# [INSTALL] SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas pyTelegramBotAPI ta psutil python-dotenv


In [ ]:
# [INFO] SECTION 2: INFRASTRUCTURE (Firebase, Telegram & Secrets)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf, warnings, psutil, time
from firebase_admin import credentials, firestore
from datetime import datetime
import telebot, ta
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try:
                raw_firebase = user_secrets.get_secret("FIREBASE_KEY_JSON")
                token = user_secrets.get_secret("TELEGRAM_TOKEN")
                chat_id = user_secrets.get_secret("TELEGRAM_CHAT_ID")
                return {"key": json.loads(raw_firebase), "token": token, "chat_id": chat_id}
            except Exception as e:
                print(f"[ERROR] Kaggle Secrets missing! Error: {e}")
                return {"key": None, "token": None, "chat_id": None}
        else:
            from dotenv import load_dotenv
            load_dotenv()
            return {
                "key": json.loads(os.getenv("FIREBASE_KEY_JSON", "{}")),
                "token": os.getenv("TELEGRAM_TOKEN"),
                "chat_id": os.getenv("TELEGRAM_CHAT_ID")
            }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            if not secrets.get('key'):
                raise ValueError("FIREBASE_KEY_JSON is missing. Check Kaggle Secrets.")
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()

    def push_task(self, queue_name: str, data):
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})

    def wait_for_queue(self, queue_name: str, max_retries=20, interval=60):
        for i in range(max_retries):
            docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
            if docs:
                tasks = []
                for doc in docs:
                    dt = doc.to_dict()
                    tasks.append(dt.get('payload', dt))
                    doc.reference.delete()
                return tasks
            if i < max_retries - 1:
                print(f"[WAIT] {queue_name} queue empty. Retrying ({i+1}/{max_retries}) in {interval}s...", flush=True)
                time.sleep(interval)
        return []
        
    def get_latest_history(self, limit=10):
        docs = self.db.collection("glu_stock_history").order_by("timestamp", direction=firestore.Query.DESCENDING).limit(limit).get()
        return [doc.to_dict() for doc in docs]

    def get_active_trades(self):
        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()
        return [(doc.id, doc.to_dict()) for doc in docs]
        
    def update_trade(self, doc_id, data):
        self.db.collection("glu_stock_trades").document(doc_id).update(data)
        
    def insert_trade(self, trade_data):
        self.db.collection("glu_stock_trades").add(trade_data)
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({
            'timestamp': datetime.now().isoformat(), 
            'phase': phase.upper(), 
            'details': details
        })


In [ ]:
# [LOGIC] SECTION 3: TRADING AGENT & RISK MANAGER
class RiskManager:
    def __init__(self, equity_risk=0.01, portfolio_equity=100000000):
        self.equity_risk = equity_risk
        self.portfolio_equity = portfolio_equity
        
    def calculate_position(self, df, ticker):
        try:
            close = df['Close'].squeeze()
            high = df['High'].squeeze()
            low = df['Low'].squeeze()
            curr_price = float(close.iloc[-1])
            atr = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range().iloc[-1]
            
            stop_loss = curr_price - (2.0 * atr)
            risk_per_share = curr_price - stop_loss
            total_risk_amt = self.portfolio_equity * self.equity_risk
            
            shares = int(total_risk_amt / (risk_per_share + 1e-7))
            return shares, stop_loss, curr_price + (3.0 * atr)
        except: return 0, 0, 0

class TradingAgent:
    def __init__(self, fb):
        self.fb = fb
        self.risk = RiskManager()
        self.logs = []

    def _log(self, phase, msg):
        self.logs.append(f"- [{phase}] {msg}")
        self.fb.log_event(phase, msg)
        print(f"[{phase}] {msg}")

    def manage_open_trades(self):
        active = self.fb.get_active_trades()
        if not active: return []
        
        results = []
        for doc_id, t in active:
            try:
                ticker = t['ticker']
                df = yf.download(ticker, period='1d', progress=False)
                curr = float(df['Close'].iloc[-1])
                
                if curr <= t['stop_loss']: reason = "SL HIT"
                elif curr >= t['take_profit']: reason = "TP HIT"
                else: 
                    results.append({'ticker': ticker, 'pnl': (curr - t['entry_price']) * t['shares'], 'curr': curr})
                    continue
                
                pnl = (curr - t['entry_price']) * t['shares']
                self.fb.update_trade(doc_id, {
                    'status': 'CLOSED', 'exit_price': curr, 
                    'exit_date': datetime.now().isoformat(), 'reason': reason, 'pnl': pnl
                })
                self._log('CLOSER', f'Closed {ticker} @ {curr} ({reason}) | PnL: {pnl:,.0f}')
            except: pass
        return results

    def execute_signals(self, signals):
        active_count = len(self.fb.get_active_trades())
        if active_count >= 10:
            self._log('BLOCK', 'Portfolio full (10 positions).')
            return
            
        for ticker, data in signals.items():
            try:
                df = yf.download(ticker, period='30d', progress=False)
                shares, sl, tp = self.risk.calculate_position(df, ticker)
                if shares > 0:
                    trade_data = {
                        'ticker': ticker, 'shares': shares, 'entry_price': data['price'],
                        'stop_loss': sl, 'take_profit': tp, 'entry_date': datetime.now().isoformat(),
                        'status': 'OPEN'
                    }
                    self.fb.insert_trade(trade_data)
                    self._log('EXECUTION', f'Opened {ticker} @ {data["price"]} | SL: {sl:,.0f}')
            except Exception as e: print(f'[WARN] Exec error {ticker}: {e}')


In [ ]:
# [RUN] SECTION 4: UNIFIED EXECUTION CYCLE
def run_full_cycle():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    agent = TradingAgent(fb)
    
    # 1. Closer
    active_results = agent.manage_open_trades()
    
    # 2. Sequential Execution
    signal_batches = fb.wait_for_queue('signals')
    if signal_batches:
        all_signals = {}
        for batch in signal_batches: all_signals.update(batch)
        agent.execute_signals(all_signals)
    else:
        print('[INFO] No new signals to process.')
    
    # 3. Final Report
    mem = psutil.virtual_memory().percent
    pnl_sum = sum([r['pnl'] for r in active_results])
    active_str = "\n".join([f"- {r['ticker']}: {r['curr']:,.0f} ({r['pnl']:+,.0f})" for r in active_results])
    activity_str = "\n".join(agent.logs)
    
    report = (
        f"**[GLU-STOCK] MISSION UPDATE**\n"
        f"**ACTIVITY**:\n{activity_str if activity_str else '- No major moves.'}\n\n"
        f"**PORTFOLIO**:\n{active_str if active_str else '- Empty.'}\n\n"
        f"**UNREALIZED PNL**: {pnl_sum:+,.0f} IDR\n"
        f"**SYSTEM**: RAM {mem}%"
    )
    
    print("--- TELEGRAM REPORT ---")
    print(report)
    
    if secrets.get('token') and secrets.get('chat_id'):
        try:
            bot = telebot.TeleBot(secrets['token'])
            bot.send_message(secrets['chat_id'], report, parse_mode="Markdown")
        except Exception as e: print(f"[ERROR] Telegram upload failed: {e}")

run_full_cycle()